# Keithley 2460 basic connectivity

Read-only commissioning probe for the instrument at `169.254.77.125`. It checks the network path and sends only `*IDN?`, `*LANG?`, and the manual-reviewed query for the current source-output state. It does **not** reset the instrument, alter configuration, start a measurement, upload code, or control the source output.

Before running, confirm at the front panel that the OUTPUT indicator is off. A successful identity query is not a battery-safety or hardware-acceptance result.

The commands and ports come from the official [Model 2460 Reference Manual, revision C](https://download.tek.com/manual/2460-901-01C_Sept_2019_Ref.pdf), sections 2, 12, and 15.

## Environment

From the repository root, prepare the locked environment with:

```bash
uv sync --locked --python 3.12 --extra visa
```

Then select the repository's `.venv` as the notebook kernel. Automatic VISA discovery is not required because the link-local instrument is opened through explicit resource names.

In [ ]:
import platform
import sys
from importlib.metadata import version

print(f"python={sys.version.split()[0]}")
print(f"platform={platform.platform()}")
for package in ("pyvisa", "pyvisa-py", "ophyd-electrochemistry"):
    print(f"{package}={version(package)}")

## Network reachability

On macOS, the instrument's link-local address should be reached through an active Ethernet interface with a `169.254.x.x/16` address. The TCP checks only establish and close connections; they send no instrument commands. Port 4880 is included to document that HiSLIP is not available on the tested unit. Do not probe port 5030: the manual defines it as a dead-socket termination service.

In [ ]:
import socket
import subprocess
from datetime import datetime

HOST = "169.254.77.125"
PORTS = {
    80: "HTTP instrument web interface",
    111: "RPC port mapper used by VXI-11",
    1024: "VXI-11",
    5025: "raw instrument socket",
    4880: "HiSLIP (not expected on this unit)",
}

print(f"probe_time={datetime.now().astimezone().isoformat()}")
ping = subprocess.run(
    ["ping", "-c", "3", "-W", "1000", HOST],
    capture_output=True,
    text=True,
    timeout=8,
    check=False,
)
print(ping.stdout if ping.stdout else ping.stderr)

for port, description in PORTS.items():
    try:
        with socket.create_connection((HOST, port), timeout=2):
            state = "open"
    except OSError as exc:
        state = f"not open ({exc})"
    print(f"tcp/{port}: {state} — {description}")

## Bounded commissioning queries

The raw-socket session explicitly sets newline framing. Each resource is opened, queried, and closed before the next is tried so there are no competing control sessions. The 3-second timeouts bound connection and response waits.

In [ ]:
import json

from ophyd_electrochemistry.keithley.k2460.commissioning import (
    run_read_only_diagnostic,
)
from ophyd_electrochemistry.keithley.k2460.transport import VisaTransportConfig

RESOURCES = (
    "TCPIP0::169.254.77.125::5025::SOCKET",
    "TCPIP0::169.254.77.125::inst0::INSTR",
)

results = [
    run_read_only_diagnostic(VisaTransportConfig(resource_name=resource, backend="@py"))
    for resource in RESOURCES
]
for result in results:
    print(json.dumps(result.as_dict(), indent=2, sort_keys=True))

In [ ]:
expected_identity_prefix = "KEITHLEY INSTRUMENTS,MODEL 2460,"
for result in results:
    assert result.identity_raw.startswith(expected_identity_prefix), result
    assert result.command_language in {"SCPI", "TSP"}, result
    assert result.mutating_commands_sent is False
    assert result.output_state_checked is True
    assert result.output_enabled is False
    print(f"PASS: {result.resource_name} -> {result.identity_raw} ({result.command_language})")

## Results observed on 2026-10-05

- Mac Ethernet interface `en11`: `169.254.109.23/16`, 1000baseT full duplex.
- ARP resolved `169.254.77.125` to `08:00:11:27:17:22`.
- ICMP: 3/3 replies, 0% loss, 1.298–1.722 ms.
- TCP open: 80, 111, 1024, and 5025. TCP 4880 refused the connection.
- Raw-socket and VXI-11 VISA resources both returned `KEITHLEY INSTRUMENTS,MODEL 2460,04686198,1.7.16a`.
- Active command language: `SCPI`; `:OUTPut:STATe?` returned `0` (OFF).

This passes the initial reachability, identity, and instrument-reported output-OFF check only. It does not complete hardware gate G05, prove electrical isolation or OFF-mode behavior, or validate any sourcing/measurement behavior with the coin cell.